# Setup

In [ ]:
import os
import json

from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_DEFAULT_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_MODEL = "ibm-granite/granite-4.1-8b"

In [ ]:
import langchain_openrouter

from langchain_core.messages import HumanMessage
from langchain_core.messages import SystemMessage
from langchain_core.messages import AIMessage

from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder

from langchain_core.output_parsers import JsonOutputParser
from langchain_core.output_parsers import CommaSeparatedListOutputParser

from pydantic import BaseModel, Field

In [ ]:
## list all the parameters that can be used to create a chat model
help(langchain_openrouter.ChatOpenRouter)

# Create Model

In [ ]:
def llm_model(params=None):

    # 1. Define sensible defaults
    config = {
        "model": OPENROUTER_MODEL,
        "api_key": OPENROUTER_API_KEY,
        "base_url": OPENROUTER_DEFAULT_BASE_URL,
        "temperature": 0.5,
        "max_tokens": 256,
        "max_completion_tokens": 128
    }
    
    if params:
        config.update(params)
        
    # 3. Initialize the model
    model = langchain_openrouter.ChatOpenRouter(
        model=config["model"],
        api_key=config["api_key"],
        base_url=config["base_url"],
        temperature=config["temperature"],
        max_tokens=config["max_tokens"],
        max_completion_tokens=config["max_completion_tokens"]
    )

    return model

def llm_model_response(prompt_text, params=None):
            
    # 3. Initialize the model
    model = llm_model(params)

    response = model.invoke(prompt_text)

    return response

# Langchain Concepts

## Chat Message

In [ ]:
from prompt_toolkit import prompt


OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

msg = model.invoke(
    [
        SystemMessage(content="You are a helpful AI bot that assists a user in choosing the perfect book to read in one short sentence"),
        HumanMessage(content="I enjoy mystery novels, what should I read?")
    ]
)

print(json.dumps(msg.response_metadata, indent=4))
print(msg.content)


In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

msg = model.invoke(
    [
        SystemMessage(content="You are a supportive AI bot that suggests fitness activities to a user in one short sentence"),
        HumanMessage(content="I like high-intensity workouts, what should I do?"),
        AIMessage(content="You should try a CrossFit class"),
        HumanMessage(content="How often should I attend?")
    ]
)

print(json.dumps(msg.response_metadata, indent=4))
print(msg.content)

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

msg = model.invoke(
    [
        # can also exclude SystemMessage and it will default to a helpful assistant
        #SystemMessage(content="You are a supportive AI bot that suggests fitness activities to a user in one short sentence"),
        HumanMessage(content="I like high-intensity workouts, what should I do?"),
        AIMessage(content="You should try a CrossFit class"),
        HumanMessage(content="How often should I attend?")
    ]
)

print(json.dumps(msg.response_metadata, indent=4))
print(msg.content)

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

# creative
params_creative = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 512,
    "max_completion_tokens": 256
}

# precise
params_precise = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,  
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.2,
    "max_tokens": 512,
    "max_completion_tokens": 256
}

model_creative = llm_model(params=params_creative)
model_precise = llm_model(params=params_precise)

prompts = [
    "Write a short poem about artificial intelligence",
    "What are the key components of a neural network?",
    "List 5 tips for effective time management"
]

for prompt in prompts:
    response_creative = model_creative.invoke(prompt)
    response_precise = model_precise.invoke(prompt)

    print(f"Prompt: {prompt}")
    print(f"Creative Response: {response_creative.text}")
    print(f"Precise Response: {response_precise.text}")
    print("-" * 50)



## Prompt Templates

### String prompt templates

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

prompt = PromptTemplate.from_template("Tell me one {adjective} joke about {topic}")

input = {"adjective": "funny", "topic": "cats"} 

prompt.invoke(input)

chain = prompt | model
response = chain.invoke(input = input)

print(json.dumps(response.response_metadata, indent=4))
print(response.content)

### Chat prompt templates

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

# Create a ChatPromptTemplate with a list of message tuples
# Each tuple contains a role ("system" or "user") and the message content
# The system message sets the behavior of the assistant
# The user message includes a variable placeholder {topic} that will be replaced later
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "Tell me a joke about {topic}")
])

# Create a dictionary with the variable to be inserted into the template
# The key "topic" matches the placeholder name in the user message
input = {"topic": "cats"}

# Format the chat template with our input values
# This replaces {topic} with "cats" in the user message
# The result will be a formatted chat message structure ready to be sent to a model
prompt.invoke(input)

chain = prompt | model
response = chain.invoke(input = input)

print(json.dumps(response.response_metadata, indent=4))
print(response.content)

### Messages Placeholder

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

# Create a ChatPromptTemplate with a system message and a placeholder for multiple messages
# The system message sets the behavior for the assistant
# MessagesPlaceholder allows for inserting multiple messages at once into the template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder("msgs")  
])

# Create an input dictionary where the key matches the MessagesPlaceholder name
# The value is a list of message objects that will replace the placeholder
# Here we're adding a single HumanMessage asking about the day after Tuesday
input = {"msgs": [HumanMessage(content="What is the day after Tuesday?")]}

# Format the chat template with our input dictionary
# This replaces the MessagesPlaceholder with the HumanMessage in our input
# The result will be a formatted chat structure with a system message and our human message
prompt.invoke(input)

chain = prompt | model
response = chain.invoke(input = input)

print(json.dumps(response.response_metadata, indent=4))
print(response.content)

## Output Parsers

### JSON parser

In [ ]:
# Define your desired data structure.
class Joke(BaseModel):
    setup: str = Field(description="question to set up a joke")
    punchline: str = Field(description="answer to resolve the joke")

OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

joke_query = "Tell me a joke."

# Set up a parser + inject instructions into the prompt template.
output_parser = JsonOutputParser(pydantic_object=Joke)

# Get the formatting instructions for the output parser
# This generates guidance text that tells the LLM how to format its response
format_instructions = output_parser.get_format_instructions()

# Create a prompt template that includes:
# 1. Instructions for the LLM to answer the user's query
# 2. Format instructions to ensure the LLM returns properly structured data
# 3. The actual user query placeholder
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],  # Dynamic variables that will be provided when invoking the chain
    partial_variables={"format_instructions": format_instructions},  # Static variables set once when creating the prompt
)

# Create a processing chain that:
# 1. Formats the prompt using the template
# 2. Sends the formatted prompt to the LLM
# 3. Parses the LLM's response using the output parser to extract structured data
chain = prompt | model | output_parser

# Invoke the chain with a specific query about jokes
# This will:
# 1. Format the prompt with the joke query
# 2. Send it to the LLM
# 3. Parse the response into the structure defined by your output parser
# 4. Return the structured result
response = chain.invoke({"query": joke_query})

print(json.dumps(response, indent=4))

### Comma-separated list parser

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

# Create an instance of the parser that will convert comma-separated text into a Python list
output_parser = CommaSeparatedListOutputParser()

# Get formatting instructions that will tell the LLM how to structure its response
# These instructions explain to the LLM that it should return items in a comma-separated format
format_instructions = output_parser.get_format_instructions()

# Create a prompt template that:
# 1. Instructs the LLM to answer the user query
# 2. Includes format instructions so the LLM knows to respond with comma-separated values
# 3. Asks the LLM to list five items of the specified subject
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{subject}\n",
    input_variables=["subject"],  # Dynamic variables that will be provided when invoking the chain
    partial_variables={"format_instructions": format_instructions},  # Static variables set once when creating the prompt
)

# Create a processing chain that:
# 1. Formats the prompt using the template
# 2. Sends the formatted prompt to the LLM
# 3. Parses the LLM's response using the output parser to extract structured data
chain = prompt | model | output_parser

# Invoke the chain with a specific query about jokes
# This will:
# 1. Format the prompt with the joke query
# 2. Send it to the LLM
# 3. Parse the response into the structure defined by your output parser
# 4. Return the structured result
response = chain.invoke({"subject": "ice cream flavors"})

print(response)